# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template and guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing all schema entities by their `@id`.

### Dataset Source
This dataset is defined by a Croissant schema and is available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`  

Proceed step by step:
- Explore metadata and structure via schema `@id` references
- Load and inspect record sets and fields
- Extract records, process, and visualize data

*All object and column references use their full `@id` for clarity and reproducibility, as per FAIR practices.*

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and examine high-level information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show basic metadata (access attributes directly, not by subscript)
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This gives you the structure of tables and columns.

All main schema entities are referenced by their `@id`.

In [ ]:
# List RecordSets by @id and fields by @id
record_sets = [rs for rs in dataset.record_sets()]

print("Available RecordSets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs['name']})")
    print("  Fields:")
    for field in rs['fields']:
        field_id = field['@id']
        field_name = field.get('name', '')
        print(f"    - {field_id} (name: {field_name})")
    print()

## 3. Data Extraction
Load data from chosen record set(s) into a DataFrame for further analysis.

Below, we use the main record set (by its `@id`) for demonstration. *Replace or extend with additional record sets as identified in step 2 if desired.*

In [ ]:
# Choose RecordSet for analysis (replace or add more as desired)
# Inspection in step 2 will reveal actual @id values. Assume the main data table is '@id': 'https://sen.science/doi/10.71728/senscience.vq4a-28xa#main-recordset'
# If the overview cell above prints a different @id (e.g., with '/recordsets/' or '#rs1'), update below accordingly.
main_recordset_id = None
for rs in record_sets:
    if ('protein' in rs['name'].lower() or 'main' in rs['@id'].split('#')[-1]):
        main_recordset_id = rs['@id']
        break
if main_recordset_id is None:
    # Default: pick the first RecordSet
    main_recordset_id = record_sets[0]['@id']

print(f"Using RecordSet @id: {main_recordset_id}")

# List all selected record sets
selected_record_sets = [main_recordset_id]
dataframes = {}

for rec_set_id in selected_record_sets:
    # records() yields dicts keyed by field @id
    recs = list(dataset.records(record_set=rec_set_id))
    df = pd.DataFrame(recs)
    dataframes[rec_set_id] = df

# Show extracted fields (columns)
print(f"Extracted columns for RecordSet {main_recordset_id}:")
print(list(dataframes[main_recordset_id].columns))

# Quick view of the records
dataframes[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data filtering and preparation steps:
- Filter records according to a numeric field (referenced by its field `@id`)
- Normalize a numeric column
- Optionally group by a categorical field

*Modify the field `@id`s below to reflect your record set structure.*

In [ ]:
df = dataframes[main_recordset_id]

# Inspect all available columns (should all be field @id)
print('Available columns in DataFrame:')
for c in df.columns:
    print(f"- {c}")

# Choose a numeric field by its field @id. For example, @id containing "molecular_weight" or "abundance".
numeric_field_id = None
for col in df.columns:
    if "abundance" in col.lower() or "molecular_weight" in col.lower() or "mw" == col.lower():
        numeric_field_id = col
        break
# Default: just pick the first float/numeric column
if numeric_field_id is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print(f"Selected numeric field for analysis: {numeric_field_id}")

# Try to make the field numeric if loaded as string
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter: select rows with field > threshold
threshold = df[numeric_field_id].quantile(0.9) if df[numeric_field_id].notnull().sum() > 0 else 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optional: group by a textual/categorical field by @id (e.g. 'accession' or 'description')
group_field = None
for c in df.columns:
    if "accession" in c.lower() or "peptide" in c.lower() or "description" in c.lower():
        group_field = c
        break

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"Grouped mean of {numeric_field_id} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distribution of the numeric field and/or its relationship with groupings.

*Requires matplotlib or seaborn. Plots field data using their field `@id`.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Boxplot by group field (if available)
if group_field and group_field in df.columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} grouped by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, you explored the FAIR² dataset via its Croissant schema, referencing all objects by `@id`. You have:
- Loaded and validated the main dataset record set and its schema structure
- Examined and processed data by field `@id`
- Performed simple filtering, normalization, and visualized numeric field distributions

*For advanced use, continue with further statistical analyses, machine learning models, or add more RecordSets as needed.*